In [ ]:
!pip install pyspark pandas pyarrow -q

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
import json
import datetime
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("HM_Export_Serving_JSON")
    .config("spark.driver.memory", "8g")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "2g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

print("Spark da khoi tao xong.")

Mounted at /content/drive
Spark da khoi tao xong.


In [ ]:
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"

PRED_PATH = BASE_PATH + "outputs_v2/predictions/test_predictions_lgbm.parquet"
ARTICLES_PATH = BASE_PATH + "processed_v2/articles_processed.parquet"
TRANS_PATH = BASE_PATH + "processed_v2/cleaned_transactions.parquet"
CUSTOMERS_PATH = BASE_PATH + "processed_v2/customers_processed.parquet"

SERVING_DIR = BASE_PATH + "outputs_v2/serving/"
os.makedirs(SERVING_DIR, exist_ok=True)

PERSONALIZED_JSON_PATH = SERVING_DIR + "personalized_recommendations_demo.json"
ARTICLES_JSON_PATH = SERVING_DIR + "articles_catalog.json"
FALLBACK_JSON_PATH = SERVING_DIR + "fallback_recommendations.json"
DEMO_CUSTOMERS_JSON_PATH = SERVING_DIR + "demo_customer_ids.json"

print("Prediction file:", PRED_PATH)
print("Articles file  :", ARTICLES_PATH)
print("Transactions   :", TRANS_PATH)
print("Customers      :", CUSTOMERS_PATH)
print("Serving dir    :", SERVING_DIR)

Prediction file: /content/drive/MyDrive/HM-DATA/outputs_v2/predictions/test_predictions_lgbm.parquet
Articles file  : /content/drive/MyDrive/HM-DATA/processed_v2/articles_processed.parquet
Transactions   : /content/drive/MyDrive/HM-DATA/processed_v2/cleaned_transactions.parquet
Customers      : /content/drive/MyDrive/HM-DATA/processed_v2/customers_processed.parquet
Serving dir    : /content/drive/MyDrive/HM-DATA/outputs_v2/serving/


In [ ]:
MAX_DEMO_CUSTOMERS = 5000
TOP_K = 12

print("So user demo:", MAX_DEMO_CUSTOMERS)
print("Top K:", TOP_K)

So user demo: 5000
Top K: 12


In [ ]:
articles = spark.read.parquet(ARTICLES_PATH)

needed_cols = [
    "article_id",
    "prod_name",
    "product_type_name",
    "colour_group_name",
    "department_name",
    "index_group_name",
    "detail_desc"
]

available_cols = articles.columns
read_cols = [col for col in needed_cols if col in available_cols]

articles_meta = articles.select(*read_cols)

for col in needed_cols:
    if col not in articles_meta.columns:
        articles_meta = articles_meta.withColumn(col, F.lit(""))

article_id_str = F.lpad(F.col("article_id").cast("string"), 10, "0")

articles_meta = (
    articles_meta
    .withColumn("article_id_str", article_id_str)
    .withColumn(
        "image_path",
        F.concat(
            F.lit("images/"),
            F.substring(F.col("article_id_str"), 1, 3),
            F.lit("/"),
            F.col("article_id_str"),
            F.lit(".jpg")
        )
    )
    .select(
        "article_id_str",
        "prod_name",
        "product_type_name",
        "colour_group_name",
        "department_name",
        "index_group_name",
        "detail_desc",
        "image_path"
    )
)

articles_pd = articles_meta.toPandas()

for col in [
    "prod_name",
    "product_type_name",
    "colour_group_name",
    "department_name",
    "index_group_name",
    "detail_desc",
    "image_path"
]:
    articles_pd[col] = articles_pd[col].fillna("")

articles_catalog = {}

for row in articles_pd.itertuples(index=False):
    articles_catalog[row.article_id_str] = {
        "article_id": row.article_id_str,
        "prod_name": row.prod_name,
        "product_type_name": row.product_type_name,
        "colour_group_name": row.colour_group_name,
        "department_name": row.department_name,
        "index_group_name": row.index_group_name,
        "detail_desc": row.detail_desc,
        "image_path": row.image_path
    }

with open(ARTICLES_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(articles_catalog, f, ensure_ascii=False)

print("Da luu:", ARTICLES_JSON_PATH)
print("So san pham:", len(articles_catalog))

Da luu: /content/drive/MyDrive/HM-DATA/outputs_v2/serving/articles_catalog.json
So san pham: 105542


In [ ]:
preds = spark.read.parquet(PRED_PATH)

demo_customers = (
    preds
    .select("customer_id")
    .dropDuplicates()
    .orderBy(F.rand(seed=42))
    .limit(MAX_DEMO_CUSTOMERS)
)

demo_customer_ids = [
    row["customer_id"]
    for row in demo_customers.collect()
]

with open(DEMO_CUSTOMERS_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(demo_customer_ids, f, ensure_ascii=False)

print("Da luu:", DEMO_CUSTOMERS_JSON_PATH)
print("So user demo:", len(demo_customer_ids))
print("Vi du user:", demo_customer_ids[:5])

Da luu: /content/drive/MyDrive/HM-DATA/outputs_v2/serving/demo_customer_ids.json
So user demo: 5000
Vi du user: ['9780b0d6f74e486c0fcfc9611f7ca0b1e34d7cac44f7d94c7eede9fb0394e351', '9a41f464d90c89aa74f61bbe443075af9dd6fe68e940587325bd2d314c55a515', '434c56476bc2fe4da41e2eabf70dfcee9ee101c3eb1bba6b71ca153f3a4a9e25', '54c0fc9d7e5b00ca53d6c186d441bce9e95de5b6d799a969b7a5dd9c4cad6221', '204183fee6f6d6fb1370a68fba9ab598bf033d60c3e4565ad129ac279b5427c0']


In [ ]:
preds_demo = preds.join(
    F.broadcast(demo_customers),
    on="customer_id",
    how="inner"
)

w_rank = Window.partitionBy("customer_id").orderBy(
    F.desc("buy_prob"),
    F.asc("article_id")
)

top12_demo = (
    preds_demo
    .withColumn("rank", F.row_number().over(w_rank))
    .filter(F.col("rank") <= TOP_K)
    .withColumn("article_id_str", F.lpad(F.col("article_id").cast("string"), 10, "0"))
    .select(
        "customer_id",
        "article_id_str",
        "rank",
        F.col("buy_prob").alias("score")
    )
)

print("So dong top12 demo:", top12_demo.count())
top12_demo.show(5, truncate=False)

So dong top12 demo: 60000
+----------------------------------------------------------------+--------------+----+-------------------+
|customer_id                                                     |article_id_str|rank|score              |
+----------------------------------------------------------------+--------------+----+-------------------+
|0028f15d12dd2a7ea7b30b120fe2aa5aa4b90817bacb97bc0d52b9efef6a6da0|0715624001    |1   |0.41527213216525   |
|0028f15d12dd2a7ea7b30b120fe2aa5aa4b90817bacb97bc0d52b9efef6a6da0|0767423011    |2   |0.3779705232291534 |
|0028f15d12dd2a7ea7b30b120fe2aa5aa4b90817bacb97bc0d52b9efef6a6da0|0714790020    |3   |0.3337187939412078 |
|0028f15d12dd2a7ea7b30b120fe2aa5aa4b90817bacb97bc0d52b9efef6a6da0|0809238001    |4   |0.33180620798504257|
|0028f15d12dd2a7ea7b30b120fe2aa5aa4b90817bacb97bc0d52b9efef6a6da0|0905518001    |5   |0.3155855833476548 |
+----------------------------------------------------------------+--------------+----+-------------------+
only showin

In [ ]:
top12_pd = top12_demo.toPandas()

personalized_recommendations = {}

for customer_id, group in top12_pd.groupby("customer_id"):
    group = group.sort_values("rank")

    items = []
    for row in group.itertuples(index=False):
        items.append({
            "article_id": row.article_id_str,
            "rank": int(row.rank),
            "score": float(row.score),
            "method": "personalized_lgbm"
        })

    personalized_recommendations[customer_id] = items

with open(PERSONALIZED_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(personalized_recommendations, f, ensure_ascii=False)

print("Da luu:", PERSONALIZED_JSON_PATH)
print("So user co goi y:", len(personalized_recommendations))

Da luu: /content/drive/MyDrive/HM-DATA/outputs_v2/serving/personalized_recommendations_demo.json
So user co goi y: 5000


In [ ]:
customers = spark.read.parquet(CUSTOMERS_PATH)

customers_age = (
    customers
    .select("customer_id", "age")
    .withColumn(
        "age_group",
        F.when(F.col("age").isNull(), "Unknown")
         .when(F.col("age") < 25, "<25")
         .when((F.col("age") >= 25) & (F.col("age") <= 35), "25-35")
         .when((F.col("age") >= 36) & (F.col("age") <= 45), "36-45")
         .when((F.col("age") >= 46) & (F.col("age") <= 55), "46-55")
         .otherwise(">55")
    )
    .select("customer_id", "age_group")
)

customers_age.show(5, truncate=False)

+----------------------------------------------------------------+---------+
|customer_id                                                     |age_group|
+----------------------------------------------------------------+---------+
|416e62a7bd3cc5b60d56d4d35718be2b3881e44a4ae46cef136a214d721f5a52|46-55    |
|416e761fd2398a8a4fccc61f1468563465396169eb945b74f17bfe5f378fbc18|46-55    |
|416e8a4593300575bdbd7afffecf8e5a61a4b7790f4b614fe263bad09469f550|>55      |
|416e8bc1a4b7edc14b6cc104af38ab000e7371406147d9c91deab449d1d34a44|46-55    |
|416e8c899206a9e6e65fab04d64cc00d005427dc3f4fcd683a189f42362c7651|<25      |
+----------------------------------------------------------------+---------+
only showing top 5 rows


In [ ]:
transactions = spark.read.parquet(TRANS_PATH)

max_date = transactions.select(F.max("t_dat_date")).collect()[0][0]
future_start = max_date + datetime.timedelta(days=1)
decay_start = future_start - datetime.timedelta(days=21)

hist_recent = (
    transactions
    .filter((F.col("t_dat_date") >= F.lit(decay_start)) & (F.col("t_dat_date") < F.lit(future_start)))
    .withColumn("days_ago", F.datediff(F.lit(future_start), F.col("t_dat_date")))
    .withColumn("weight", F.pow(F.lit(0.95), F.col("days_ago")))
)

global_top = (
    hist_recent
    .groupBy("article_id")
    .agg(F.sum("weight").alias("score"))
    .orderBy(F.desc("score"), F.asc("article_id"))
    .limit(TOP_K)
    .withColumn("rank", F.row_number().over(Window.orderBy(F.desc("score"), F.asc("article_id"))))
    .withColumn("article_id_str", F.lpad(F.col("article_id").cast("string"), 10, "0"))
    .select("article_id_str", "rank", "score")
)

global_top.show(TOP_K, truncate=False)

+--------------+----+------------------+
|article_id_str|rank|score             |
+--------------+----+------------------+
|0924243001    |1   |1138.1327042522535|
|0909370001    |2   |1136.5860643655917|
|0751471001    |3   |1062.3918144281513|
|0918522001    |4   |1061.8958581342654|
|0448509014    |5   |970.0354761305657 |
|0918292001    |6   |905.6283791320375 |
|0915529003    |7   |902.5855775723743 |
|0751471043    |8   |846.8771899842259 |
|0865799006    |9   |840.4411842385176 |
|0915526001    |10  |811.0565383405172 |
|0706016001    |11  |795.8243870943704 |
|0762846027    |12  |784.7969952863214 |
+--------------+----+------------------+



In [ ]:
hist_with_age = hist_recent.join(
    F.broadcast(customers_age),
    on="customer_id",
    how="left"
).fillna({"age_group": "Unknown"})

age_scores = (
    hist_with_age
    .groupBy("age_group", "article_id")
    .agg(F.sum("weight").alias("score"))
)

w_age = Window.partitionBy("age_group").orderBy(
    F.desc("score"),
    F.asc("article_id")
)

age_top = (
    age_scores
    .withColumn("rank", F.row_number().over(w_age))
    .filter(F.col("rank") <= TOP_K)
    .withColumn("article_id_str", F.lpad(F.col("article_id").cast("string"), 10, "0"))
    .select("age_group", "article_id_str", "rank", "score")
)

age_top.show(20, truncate=False)

+---------+--------------+----+------------------+
|age_group|article_id_str|rank|score             |
+---------+--------------+----+------------------+
|25-35    |0909370001    |1   |530.8666681354947 |
|25-35    |0924243001    |2   |401.965553055983  |
|25-35    |0918292001    |3   |371.96770590952707|
|25-35    |0865799006    |4   |339.8924083256185 |
|25-35    |0158340001    |5   |327.4696047149647 |
|25-35    |0863583001    |6   |313.61612296661485|
|25-35    |0448509014    |7   |311.0302591957459 |
|25-35    |0866731001    |8   |301.2376965586425 |
|25-35    |0915529003    |9   |298.4536415786476 |
|25-35    |0762846027    |10  |298.0922792218678 |
|25-35    |0751471001    |11  |282.010708360918  |
|25-35    |0706016001    |12  |269.011040317659  |
|36-45    |0909370001    |1   |134.10596264817565|
|36-45    |0915529003    |2   |108.31544901886984|
|36-45    |0751471001    |3   |107.66184138298604|
|36-45    |0768912001    |4   |91.56087558772487 |
|36-45    |0924243001    |5   |

In [ ]:
global_pd = global_top.toPandas()
age_pd = age_top.toPandas()

def rows_to_recommendations(df, method):
    result = []

    df = df.sort_values("rank")

    for row in df.itertuples(index=False):
        result.append({
            "article_id": row.article_id_str,
            "rank": int(row.rank),
            "score": float(row.score),
            "method": method
        })

    return result

fallback_recommendations = {
    "global": rows_to_recommendations(global_pd, "global_bestseller")
}

age_groups = ["<25", "25-35", "36-45", "46-55", ">55", "Unknown"]

for age_group in age_groups:
    group_df = age_pd[age_pd["age_group"] == age_group]

    if len(group_df) >= TOP_K:
        fallback_recommendations[age_group] = rows_to_recommendations(
            group_df,
            "age_group_bestseller"
        )
    else:
        fallback_recommendations[age_group] = fallback_recommendations["global"]

with open(FALLBACK_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(fallback_recommendations, f, ensure_ascii=False)

print("Da luu:", FALLBACK_JSON_PATH)

for key, value in fallback_recommendations.items():
    print(key, len(value))

Da luu: /content/drive/MyDrive/HM-DATA/outputs_v2/serving/fallback_recommendations.json
global 12
<25 12
25-35 12
36-45 12
46-55 12
>55 12
Unknown 12


In [ ]:
print("Serving files:")
print(PERSONALIZED_JSON_PATH)
print(ARTICLES_JSON_PATH)
print(FALLBACK_JSON_PATH)
print(DEMO_CUSTOMERS_JSON_PATH)

with open(PERSONALIZED_JSON_PATH, "r", encoding="utf-8") as f:
    personalized_check = json.load(f)

with open(ARTICLES_JSON_PATH, "r", encoding="utf-8") as f:
    articles_check = json.load(f)

with open(FALLBACK_JSON_PATH, "r", encoding="utf-8") as f:
    fallback_check = json.load(f)

sample_customer = next(iter(personalized_check.keys()))
sample_rec = personalized_check[sample_customer][0]
sample_article = sample_rec["article_id"]

print("Sample customer:", sample_customer)
print("Sample recommendation:", personalized_check[sample_customer][:3])
print("Sample article metadata:", articles_check.get(sample_article))
print("Fallback global:", fallback_check["global"][:3])

Serving files:
/content/drive/MyDrive/HM-DATA/outputs_v2/serving/personalized_recommendations_demo.json
/content/drive/MyDrive/HM-DATA/outputs_v2/serving/articles_catalog.json
/content/drive/MyDrive/HM-DATA/outputs_v2/serving/fallback_recommendations.json
/content/drive/MyDrive/HM-DATA/outputs_v2/serving/demo_customer_ids.json
Sample customer: 000c41868d0170bf1e022a985a37f52344ba14ca5c331b816982180fb4b169e1
Sample recommendation: [{'article_id': '0723469001', 'rank': 1, 'score': 0.26377379409862894, 'method': 'personalized_lgbm'}, {'article_id': '0599580047', 'rank': 2, 'score': 0.21335260665245376, 'method': 'personalized_lgbm'}, {'article_id': '0776237021', 'rank': 3, 'score': 0.19921586456254392, 'method': 'personalized_lgbm'}]
Sample article metadata: {'article_id': '0723469001', 'prod_name': 'Kelly 2pk Melbourne push ct', 'product_type_name': 'Bra', 'colour_group_name': 'Black', 'department_name': '', 'index_group_name': '', 'detail_desc': 'Push-up bras in soft cotton jersey with 